<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700;letter-spacing:.8px">LAB 07 · LEVEL 2 · UPDATING AND DELETING</div>
<div style="color:#17212b;font-size:30px;font-weight:750">Maintain current state without rewriting event history</div>
<p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px">Use a Unique Key table for current state, a sequence column for out-of-order changes, partial column updates for sparse patches, and controlled deletes for an isolated table.</p>
<span style="display:inline-block;border:1px solid #99f6e4;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:10px;font-size:12px">All writes are isolated to tables owned by this lab</span>
</div>

## Current-state contract

A row in `order_state_lab7` represents the latest state of one order. `event_version` expresses business order for the same key; it is not arrival order. The late version below must not replace a newer version.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")


In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS order_state_lab7 (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    order_amount DECIMAL(12,2) NOT NULL,
    region VARCHAR(16) NOT NULL,
    event_version BIGINT NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES (
    "replication_num"="1",
    "enable_unique_key_merge_on_write"="true",
    "function_column.sequence_col"="event_version"
)
""")
lab.execute("TRUNCATE TABLE order_state_lab7")
lab.insert("""
INSERT INTO order_state_lab7 VALUES
    (700001, 'CREATED', 100.00, 'EAST', 1),
    (700002, 'CREATED', 200.00, 'WEST', 1)
""", title="Write initial order state")
lab.insert("""
INSERT INTO order_state_lab7 VALUES
    (700001, 'PAID', 100.00, 'EAST', 2),
    (700002, 'SHIPPED', 200.00, 'WEST', 3)
""", title="Write newer complete states")
lab.insert("""
INSERT INTO order_state_lab7 VALUES
    (700001, 'CANCELLED', 100.00, 'EAST', 1)
""", title="Replay an older state")
lab.sql("""
SELECT order_id, status, order_amount, region, event_version
FROM order_state_lab7
ORDER BY order_id
""", title="Current state after an out-of-order replay")

## Sparse updates and deletion semantics

The next table is independent from the full-row example. Enable partial update for one statement, change only `status` and `event_version`, and verify that `order_amount` and `region` remain unchanged. Then compare a business soft-delete marker with SQL `DELETE` in another isolated table.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS order_partial_lab7 (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    order_amount DECIMAL(12,2) NOT NULL,
    region VARCHAR(16) NOT NULL,
    event_version BIGINT NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES (
    "replication_num"="1",
    "enable_unique_key_merge_on_write"="true",
    "function_column.sequence_col"="event_version"
)
""")
lab.execute("TRUNCATE TABLE order_partial_lab7")
lab.insert("""
INSERT INTO order_partial_lab7 VALUES (700003, 'CREATED', 300.00, 'NORTH', 1)
""", title="Seed the partial-update row")
try:
    lab.execute("SET enable_unique_key_partial_update = true")
    lab.insert("""
    INSERT INTO order_partial_lab7 (order_id, status, event_version)
    VALUES (700003, 'CANCELLED', 2)
    """, title="Apply a sparse state change")
finally:
    lab.execute("SET enable_unique_key_partial_update = false")

lab.sql("""
SELECT order_id, status, order_amount, region, event_version
FROM order_partial_lab7
ORDER BY order_id
""", title="Partial update preserves omitted fields")

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS order_delete_lab7 (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    is_deleted BOOLEAN NOT NULL DEFAULT FALSE
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "enable_unique_key_merge_on_write"="true")
""")
lab.execute("TRUNCATE TABLE order_delete_lab7")
lab.insert("""
INSERT INTO order_delete_lab7 VALUES
    (700004, 'CREATED', FALSE),
    (700005, 'CREATED', FALSE)
""", title="Seed delete semantics")
lab.execute("UPDATE order_delete_lab7 SET is_deleted = TRUE WHERE order_id = 700004")
lab.sql(
    "SELECT order_id, status, is_deleted FROM order_delete_lab7 ORDER BY order_id",
    title="Soft delete remains query-visible",
)
lab.execute("DELETE FROM order_delete_lab7 WHERE order_id = 700005")
lab.sql(
    "SELECT order_id, status, is_deleted FROM order_delete_lab7 ORDER BY order_id",
    title="SQL delete changes logical visibility",
    final=True,
)

## Takeaway

- Unique Key plus Merge-on-Write exposes the latest logical row, while old physical files are reclaimed later by compaction.
- A sequence column compares versions for the same key and prevents an old replay from winning.
- Partial update semantics are different from omitting columns in an ordinary full-row write.
- A business `is_deleted` marker and SQL `DELETE` answer different questions; neither is proof of immediate physical reclamation.